# 🧠 Mineflayer + LLM Agent v2 — Minecraft AI on Colab

Runs [Mineflayer](https://github.com/PrismarineJS/node-mineflayer) + Ollama LLM on Colab.
Inspired by **Mindcraft**, **MC-CIV**, and **Minecraft_AI** projects.

## 41+ Tools (like the video)

```
MOVEMENT:    moveTo, followPlayer, explore, stop, lookAt, jump, sprint, sneak
MINING:      collectBlocks, digBlock, mineResource, stripMine
BUILDING:    placeBlock, buildHouse, buildWall, buildFloor, towerUp, bridge
CRAFTING:    craft, smelt, enchant, repair
COMBAT:      attack, hunt, defend, flee, jumpAttack
SURVIVAL:    eat, sleep, fish, farm, tame, breed
INVENTORY:   equip, unequip, dropItem, depositItem, withdrawItem, sortInventory
SOCIAL:      chat, trade, useOn (shear/milk/bonemeal/etc)
OBSERVATION: getInventory, getNearbyBlocks, getNearbyEntities, getHealth, inspectWorld
```

**Instructions:**
1. Runtime > Change runtime type > GPU (T4 for 7B, A100 for 70B)
2. Execute all cells in order
3. Set MC server + Ollama model in Cell 2

In [ ]:
# Cell 1: Install dependencies
!curl -fsSL https://ollama.com/install.sh | sh
!pip install fastapi uvicorn pyngrok requests
!npm install mineflayer mineflayer-pathfinder mineflayer-auto-eat mineflayer-pvp mineflayer-collectblock mineflayer-tool prismarine-item prismarine-recipe minecraft-data vec3
print('✅ Dependencies installed (41+ tools ready)')

In [ ]:
# Cell 2: Configuration
MC_VERSION = '1.21.1'
MC_SERVER = ''  # <-- Your server IP:port
MC_USERNAME = 'LLM_Bot'

LLM_MODEL = 'qwen2.5:14b'  # 7b, 14b (T4) or 32b, 70b (A100)
LLM_HOST = 'http://localhost:11434'

NGROK_AUTHTOKEN = ''  # <-- Paste your ngrok authtoken
BOT_WEBHOOK_URL = ''  # <-- Bot webhook URL
API_PORT = 7000

MAX_ACTIONS_PER_GOAL = 80
ACTION_DELAY_MS = 100  # Reduced from 300 for fluid execution
OBSERVATION_RADIUS = 32
LLM_TIMEOUT = 90  # Reduced from 120 for faster failure detection
WORLD_CACHE_MS = 1500  # Cache world state to avoid redundant fetches

print(f'Config: MC {MC_VERSION}, LLM={LLM_MODEL}, Server={MC_SERVER}, delay={ACTION_DELAY_MS}ms')

In [ ]:
# Cell 3: Start Ollama + pull model
import subprocess, os, time
os.system('pkill ollama 2>/dev/null; sleep 2')
env = os.environ.copy()
env['OLLAMA_HOST'] = '0.0.0.0:11434'
ollama_proc = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, env=env)
time.sleep(3)
print(f'⏳ Pulling LLM model {LLM_MODEL}...')
os.system(f'ollama pull {LLM_MODEL}')
print(f'✅ Ollama ready with {LLM_MODEL}')

In [ ]:
# Cell 4: Mineflayer bot script v2 — 41+ tools (Mindcraft/MC-CIV style)
mineflayer_script = r'''"use strict";
const mineflayer = require('mineflayer');
const pathfinder = require('mineflayer-pathfinder');
const { Movements, goals } = pathfinder;
const autoEat = require('mineflayer-auto-eat');
const pvp = require('mineflayer-pvp');
const collectBlock = require('mineflayer-collectblock').plugin;
const { Vec3 } = require('vec3');
const mcData = require('minecraft-data');

const bot = mineflayer.createBot({
  host: process.env.MC_SERVER,
  port: parseInt(process.env.MC_PORT || '25565'),
  username: process.env.MC_USERNAME || 'LLM_Bot',
  version: process.env.MC_VERSION || '1.21.1',
  hideErrors: false,
});

let mcDataCache = null;
bot.once('spawn', () => {
  mcDataCache = mcData(bot.version);
  const defaultMove = new Movements(bot, mcDataCache);
  defaultMove.allowParkour = true;
  defaultMove.allowSprinting = true;
  defaultMove.allowDig = true;
  defaultMove.allowPlace = true;
  defaultMove.canDig = true;
  bot.pathfinder.setMovements(defaultMove);
  bot.loadPlugin(autoEat);
  bot.loadPlugin(pvp);
  bot.loadPlugin(collectBlock);
  console.log('SPAWNED');
  bot.chat('Hello! I am an LLM-controlled bot with 41+ tools.');
});

bot.on('health', () => { if (bot.health <= 0) console.log('DIED'); });
bot.on('kicked', (r) => console.log('KICKED:', r));
bot.on('error', (e) => console.log('ERROR:', e.message));
bot.on('end', () => console.log('DISCONNECTED'));

function ok(msg) { return { success: true, message: msg }; }
function fail(msg) { return { success: false, message: msg }; }

async function executeAction(action) {
  const { type, params } = action;
  const p = params || {};
  switch (type) {

    // ═══ MOVEMENT (8) ═══
    case 'moveTo': {
      const goal = new goals.GoalBlock(Math.floor(p.x), Math.floor(bot.entity.position.y), Math.floor(p.z));
      bot.pathfinder.setGoal(goal);
      await new Promise((res) => { bot.once('goal_reached', res); setTimeout(res, 30000); });
      return ok(`Arrived at ${p.x}, ${p.z}`);
    }
    case 'followPlayer': {
      const target = bot.players[p.name]?.entity || bot.nearestEntity(e => e.username === p.name);
      if (!target) return fail(`Player ${p.name} not found`);
      bot.pathfinder.setGoal(new goals.GoalFollow(target, p.distance || 3), true);
      return ok(`Following ${p.name}`);
    }
    case 'explore': {
      const r = p.radius || 50;
      const dx = (Math.random() - 0.5) * r * 2;
      const dz = (Math.random() - 0.5) * r * 2;
      const tx = Math.floor(bot.entity.position.x + dx);
      const tz = Math.floor(bot.entity.position.z + dz);
      bot.pathfinder.setGoal(new goals.GoalBlock(tx, Math.floor(bot.entity.position.y), tz));
      await new Promise((res) => { bot.once('goal_reached', res); setTimeout(res, 30000); });
      return ok(`Explored to ${tx}, ${tz}`);
    }
    case 'stop': { bot.pathfinder.setGoal(null); bot.clearControlStates(); return ok('Stopped'); }
    case 'lookAt': { await bot.lookAt(new Vec3(p.x, p.y, p.z)); return ok(`Looking at ${p.x},${p.y},${p.z}`); }
    case 'jump': { bot.setControlState('jump', true); await new Promise(r => setTimeout(r, 500)); bot.setControlState('jump', false); return ok('Jumped'); }
    case 'sprint': { bot.setControlState('sprint', p.enable !== false); return ok(`Sprint ${p.enable !== false ? 'on' : 'off'}`); }
    case 'sneak': { bot.setControlState('sneak', p.enable !== false); return ok(`Sneak ${p.enable !== false ? 'on' : 'off'}`); }

    // ═══ MINING (4) ═══
    case 'collectBlocks': {
      const blockType = mcDataCache.blocksByName[p.blockType] || mcDataCache.blocksByName[p.type];
      if (!blockType) return fail(`Unknown block: ${p.blockType || p.type}`);
      const count = Math.min(64, p.count || 1);
      const blocks = bot.findBlocks({ matching: blockType.id, maxDistance: 64, count });
      if (!blocks.length) return fail(`No ${p.blockType || p.type} found within 64 blocks`);
      let collected = 0;
      for (const pos of blocks) {
        if (collected >= count) break;
        const b = bot.blockAt(pos);
        if (!b) continue;
        try { await bot.collectBlock.collect(b); collected++; } catch (e) {}
      }
      return ok(`Collected ${collected}/${count} ${p.blockType || p.type}`);
    }
    case 'digBlock': {
      const b = bot.blockAt(new Vec3(p.x, p.y, p.z));
      if (!b || b.name === 'air') return fail('No block at position');
      await bot.dig(b);
      return ok(`Dug ${b.name} at ${p.x},${p.y},${p.z}`);
    }
    case 'mineResource': {
      const types = (p.resource || p.type).split(',').map(s => s.trim());
      for (const t of types) {
        const blockType = mcDataCache.blocksByName[t];
        if (!blockType) continue;
        const blocks = bot.findBlocks({ matching: blockType.id, maxDistance: 128, count: p.count || 1 });
        if (blocks.length) {
          let collected = 0;
          for (const pos of blocks) {
            if (collected >= (p.count || 1)) break;
            try { await bot.collectBlock.collect(bot.blockAt(pos)); collected++; } catch {}
          }
          return ok(`Mined ${collected} ${t}`);
        }
      }
      return fail(`No ${p.resource || p.type} found within 128 blocks`);
    }
    case 'stripMine': {
      const dir = p.direction || 'north';
      const length = Math.min(50, p.length || 20);
      const dirs = { north: [0,0,-1], south: [0,0,1], east: [1,0,0], west: [-1,0,0] };
      const [dx, , dz] = dirs[dir] || [0,0,-1];
      const pos = bot.entity.position;
      for (let i = 0; i < length; i++) {
        const b1 = bot.blockAt(pos.offset(dx*i, 1, dz*i));
        const b2 = bot.blockAt(pos.offset(dx*i, 0, dz*i));
        if (b1 && b1.name !== 'air') try { await bot.dig(b1); } catch {}
        if (b2 && b2.name !== 'air') try { await bot.dig(b2); } catch {}
      }
      return ok(`Strip mined ${length} blocks ${dir}`);
    }

    // ═══ BUILDING (6) ═══
    case 'placeBlock': {
      const item = bot.inventory.items().find(i => i.name === p.blockType || i.name === p.type);
      if (!item) return fail(`No ${p.blockType || p.type} in inventory`);
      await bot.equip(item, 'hand');
      const refBlock = bot.blockAt(new Vec3(p.x, p.y - 1, p.z));
      if (!refBlock) return fail('No reference block below target');
      await bot.placeBlock(refBlock, new Vec3(0, 1, 0));
      return ok(`Placed ${p.blockType || p.type} at ${p.x},${p.y},${p.z}`);
    }
    case 'buildHouse': {
      const size = p.size || 5;
      const pos = bot.entity.position;
      const material = p.material || 'oak_planks';
      const item = bot.inventory.items().find(i => i.name === material);
      if (!item) return fail(`No ${material} — collect wood first`);
      const sx = Math.floor(pos.x), sy = Math.floor(pos.y), sz = Math.floor(pos.z);
      let placed = 0;
      await bot.equip(item, 'hand');
      for (let x = 0; x < size; x++) for (let z = 0; z < size; z++) {
        const ref = bot.blockAt(new Vec3(sx+x, sy-1, sz+z));
        if (ref) try { await bot.placeBlock(ref, new Vec3(0,1,0)); placed++; } catch {}
      }
      for (let h = 0; h < 3; h++) {
        for (let x = 0; x < size; x++) { for (const z of [0, size-1]) {
          const ref = bot.blockAt(new Vec3(sx+x, sy+h, sz+z));
          if (ref) try { await bot.placeBlock(ref, new Vec3(0,1,0)); placed++; } catch {}
        }}
        for (let z = 0; z < size; z++) { for (const x of [0, size-1]) {
          const ref = bot.blockAt(new Vec3(sx+x, sy+h, sz+z));
          if (ref) try { await bot.placeBlock(ref, new Vec3(0,1,0)); placed++; } catch {}
        }}
      }
      return ok(`Built ${size}x${size} house (${placed} blocks)`);
    }
    case 'buildWall': {
      const length = Math.min(30, p.length || 10), height = Math.min(5, p.height || 3);
      const material = p.material || 'cobblestone';
      const item = bot.inventory.items().find(i => i.name === material);
      if (!item) return fail(`No ${material}`);
      const pos = bot.entity.position;
      await bot.equip(item, 'hand');
      let placed = 0;
      for (let l = 0; l < length; l++) for (let h = 0; h < height; h++) {
        const ref = bot.blockAt(new Vec3(Math.floor(pos.x)+l, Math.floor(pos.y)+h-1, Math.floor(pos.z)));
        if (ref) try { await bot.placeBlock(ref, new Vec3(0,1,0)); placed++; } catch {}
      }
      return ok(`Built wall ${length}x${height} (${placed} blocks)`);
    }
    case 'buildFloor': {
      const size = p.size || 5;
      const material = p.material || 'oak_planks';
      const item = bot.inventory.items().find(i => i.name === material);
      if (!item) return fail(`No ${material}`);
      const pos = bot.entity.position;
      await bot.equip(item, 'hand');
      let placed = 0;
      for (let x = 0; x < size; x++) for (let z = 0; z < size; z++) {
        const ref = bot.blockAt(new Vec3(Math.floor(pos.x)+x, Math.floor(pos.y)-1, Math.floor(pos.z)+z));
        if (ref) try { await bot.placeBlock(ref, new Vec3(0,1,0)); placed++; } catch {}
      }
      return ok(`Built ${size}x${size} floor (${placed} blocks)`);
    }
    case 'towerUp': {
      const height = Math.min(20, p.height || 5);
      const material = p.material || 'dirt';
      const item = bot.inventory.items().find(i => i.name === material);
      if (!item) return fail(`No ${material}`);
      const pos = bot.entity.position;
      await bot.equip(item, 'hand');
      let placed = 0;
      for (let h = 0; h < height; h++) {
        const ref = bot.blockAt(new Vec3(Math.floor(pos.x), Math.floor(pos.y)+h-1, Math.floor(pos.z)));
        if (ref) try { await bot.placeBlock(ref, new Vec3(0,1,0)); placed++; } catch {}
        bot.setControlState('jump', true); await new Promise(r => setTimeout(r, 300)); bot.setControlState('jump', false);
      }
      return ok(`Towered up ${height} (${placed} placed)`);
    }
    case 'bridge': {
      const length = Math.min(30, p.length || 10);
      const material = p.material || 'cobblestone';
      const item = bot.inventory.items().find(i => i.name === material);
      if (!item) return fail(`No ${material}`);
      const pos = bot.entity.position;
      await bot.equip(item, 'hand');
      bot.setControlState('sneak', true);
      let placed = 0;
      for (let l = 0; l < length; l++) {
        const ref = bot.blockAt(new Vec3(Math.floor(pos.x), Math.floor(pos.y)-1, Math.floor(pos.z)-l));
        if (ref) try { await bot.placeBlock(ref, new Vec3(0,0,-1)); placed++; } catch {}
      }
      bot.setControlState('sneak', false);
      return ok(`Bridged ${length} (${placed} placed)`);
    }

    // ═══ CRAFTING (4) ═══
    case 'craft': {
      const itemName = p.item || p.name;
      const itemData = mcDataCache.itemsByName[itemName];
      if (!itemData) return fail(`Unknown item: ${itemName}`);
      const recipes = bot.recipesFor(itemData.id);
      if (!recipes.length) return fail(`No recipe for ${itemName} — missing resources`);
      const count = Math.min(16, p.count || 1);
      await bot.craft(recipes[0], count, null);
      return ok(`Crafted ${count}x ${itemName}`);
    }
    case 'smelt': {
      const fuel = p.fuel || 'coal';
      const count = Math.min(32, p.count || 1);
      let furnaceBlock = bot.findBlock({ matching: mcDataCache.blocksByName['furnace']?.id, maxDistance: 32 });
      if (!furnaceBlock) return fail('No furnace nearby — craft one first');
      const furnace = await bot.openFurnace(furnaceBlock);
      const fuelItem = bot.inventory.items().find(i => i.name === fuel);
      if (!fuelItem) { furnace.close(); return fail(`No ${fuel} for fuel`); }
      await furnace.putFuel(fuelItem);
      const inputItem = bot.inventory.items().find(i => i.name === (p.item || p.name));
      if (!inputItem) { furnace.close(); return fail(`No ${p.item || p.name} to smelt`); }
      for (let i = 0; i < count; i++) { await furnace.putInput(inputItem); await new Promise(r => setTimeout(r, 10000)); }
      furnace.close();
      return ok(`Smelted ${count}x ${p.item || p.name}`);
    }
    case 'enchant': return ok(`Enchant not fully implemented — need enchanting table UI`);
    case 'repair': return ok(`Repair not fully implemented — need anvil UI`);

    // ═══ COMBAT (5) ═══
    case 'attack': {
      const target = bot.nearestEntity(e => e.name === p.target || e.username === p.target || (e.name && e.name.includes(p.target)));
      if (!target) return fail(`No ${p.target} nearby`);
      bot.pvp.attack(target);
      return ok(`Attacking ${target.name || target.username}`);
    }
    case 'hunt': {
      const prey = ['cow', 'pig', 'sheep', 'chicken', 'rabbit'];
      const target = bot.nearestEntity(e => prey.includes(e.name));
      if (!target) return fail('No animals nearby to hunt');
      bot.pvp.attack(target);
      return ok(`Hunting ${target.name}`);
    }
    case 'defend': {
      const hostile = ['zombie', 'skeleton', 'creeper', 'spider', 'enderman', 'witch'];
      const target = bot.nearestEntity(e => hostile.includes(e.name));
      if (!target) return fail('No hostile mobs nearby');
      bot.pvp.attack(target);
      return ok(`Defending against ${target.name}`);
    }
    case 'flee': {
      const threat = bot.nearestEntity(e => ['zombie','skeleton','creeper','spider','enderman','witch','blaze','ghast'].includes(e.name));
      if (!threat) return fail('No threats nearby');
      const away = bot.entity.position.subtract(threat.position).normalize().scale(20);
      bot.pathfinder.setGoal(new goals.GoalBlock(Math.floor(bot.entity.position.x + away.x), Math.floor(bot.entity.position.y), Math.floor(bot.entity.position.z + away.z)));
      return ok(`Fleeing from ${threat.name}`);
    }
    case 'jumpAttack': {
      const target = bot.nearestEntity(e => e.name === p.target || e.username === p.target);
      if (!target) return fail(`No ${p.target} nearby`);
      bot.setControlState('jump', true); bot.pvp.attack(target);
      await new Promise(r => setTimeout(r, 300)); bot.setControlState('jump', false);
      return ok(`Jump-attacked ${target.name || target.username}`);
    }

    // ═══ SURVIVAL (6) ═══
    case 'eat': {
      const food = bot.inventory.items().find(i => ['bread','cooked_beef','cooked_porkchop','cooked_chicken','apple','carrot','baked_potato','cooked_mutton','cooked_rabbit','melon_slice','sweet_berries'].includes(i.name));
      if (!food) return fail('No food in inventory');
      await bot.equip(food, 'hand'); await bot.consume();
      return ok(`Ate ${food.name}`);
    }
    case 'sleep': {
      const bed = bot.findBlock({ matching: b => b.name.includes('bed'), maxDistance: 16 });
      if (!bed) return fail('No bed nearby');
      await bot.sleep(bed); return ok('Sleeping');
    }
    case 'fish': {
      const rod = bot.inventory.items().find(i => i.name === 'fishing_rod');
      if (!rod) return fail('No fishing rod');
      await bot.equip(rod, 'hand'); await bot.fish(); return ok('Fished');
    }
    case 'farm': {
      const crops = ['wheat','carrots','potatoes','beetroots'];
      let harvested = 0;
      for (const crop of crops) {
        const blockType = mcDataCache.blocksByName[crop]; if (!blockType) continue;
        const blocks = bot.findBlocks({ matching: blockType.id, maxDistance: 16 });
        for (const pos of blocks) { const b = bot.blockAt(pos); if (b && b.metadata >= 7) { try { await bot.dig(b); harvested++; } catch {} } }
      }
      return ok(`Harvested ${harvested} crops`);
    }
    case 'tame': {
      const animal = bot.nearestEntity(e => ['horse','wolf','cat','parrot'].includes(e.name));
      if (!animal) return fail('No tameable animal nearby');
      await animal.activate(); return ok(`Taming ${animal.name}`);
    }
    case 'breed': {
      const foodMap = { cow: 'wheat', pig: 'carrot', sheep: 'wheat', chicken: 'seeds', horse: 'golden_apple' };
      const foodName = foodMap[p.animal || 'cow']; if (!foodName) return fail(`Unknown animal: ${p.animal}`);
      const food = bot.inventory.items().find(i => i.name === foodName);
      if (!food) return fail(`No ${foodName}`);
      const animals = Object.values(bot.entities).filter(e => e.name === (p.animal || 'cow') && e.position.distanceTo(bot.entity.position) < 10);
      if (animals.length < 2) return fail(`Need 2 ${p.animal}s nearby`);
      await bot.equip(food, 'hand');
      for (const a of animals.slice(0, 2)) { try { await a.activate(); } catch {} }
      return ok(`Breeding ${p.animal}s`);
    }

    // ═══ INVENTORY (6) ═══
    case 'equip': {
      const item = bot.inventory.items().find(i => i.name === (p.item || p.name));
      if (!item) return fail(`No ${p.item || p.name}`);
      await bot.equip(item, p.slot || 'hand'); return ok(`Equipped ${p.item || p.name}`);
    }
    case 'unequip': { await bot.unequip(p.slot || 'hand'); return ok(`Unequipped ${p.slot || 'hand'}`); }
    case 'dropItem': {
      const item = bot.inventory.items().find(i => i.name === (p.item || p.name));
      if (!item) return fail(`No ${p.item || p.name}`);
      await bot.toss(item.type, item.metadata, Math.min(item.count, p.count || 1));
      return ok(`Dropped ${p.count || 1}x ${p.item || p.name}`);
    }
    case 'depositItem': {
      const chest = bot.findBlock({ matching: mcDataCache.blocksByName['chest']?.id, maxDistance: 16 });
      if (!chest) return fail('No chest nearby');
      const c = await bot.openChest(chest);
      const item = bot.inventory.items().find(i => i.name === (p.item || p.name));
      if (!item) { c.close(); return fail(`No ${p.item || p.name}`); }
      await c.deposit(item.type, item.metadata, Math.min(item.count, p.count || 1)); c.close();
      return ok(`Deposited ${p.count || 1}x ${p.item || p.name}`);
    }
    case 'withdrawItem': {
      const chest = bot.findBlock({ matching: mcDataCache.blocksByName['chest']?.id, maxDistance: 16 });
      if (!chest) return fail('No chest nearby');
      const c = await bot.openChest(chest);
      const item = c.items().find(i => i.name === (p.item || p.name));
      if (!item) { c.close(); return fail(`No ${p.item || p.name} in chest`); }
      await c.withdraw(item.type, item.metadata, Math.min(item.count, p.count || 1)); c.close();
      return ok(`Withdrew ${p.count || 1}x ${p.item || p.name}`);
    }
    case 'sortInventory': {
      const armor = { helmet: ['diamond_helmet','iron_helmet','golden_helmet','leather_helmet'], chestplate: ['diamond_chestplate','iron_chestplate','golden_chestplate','leather_chestplate'], leggings: ['diamond_leggings','iron_leggings','golden_leggings','leather_leggings'], boots: ['diamond_boots','iron_boots','golden_boots','leather_boots'] };
      let equipped = 0;
      for (const [slot, items] of Object.entries(armor)) {
        for (const name of items) {
          const item = bot.inventory.items().find(i => i.name === name);
          if (item) { try { await bot.equip(item, slot); equipped++; } catch {} break; }
        }
      }
      return ok(`Equipped ${equipped} armor pieces`);
    }

    // ═══ SOCIAL (2) ═══
    case 'chat': { bot.chat(p.message || ''); return ok(`Said: ${p.message}`); }
    case 'trade': {
      const villager = bot.nearestEntity(e => e.name === 'villager');
      if (!villager) return fail('No villager nearby');
      const v = await bot.openVillager(villager);
      const trade = v.trades[p.tradeIndex || 0];
      if (!trade) { v.close(); return fail('No trade available'); }
      await v.trade(trade, p.count || 1); v.close();
      return ok('Traded with villager');
    }

    // ═══ USE ON (1) ═══
    case 'useOn': {
      if (p.tool) { const tool = bot.inventory.items().find(i => i.name === p.tool); if (tool) await bot.equip(tool, 'hand'); }
      const target = bot.nearestEntity(e => e.name === p.target || e.username === p.target);
      if (target) { await target.activate(); return ok(`Used ${p.tool || 'hand'} on ${target.name}`); }
      if (p.x !== undefined) { const block = bot.blockAt(new Vec3(p.x, p.y, p.z)); if (block) { await bot.activateBlock(block); return ok(`Used ${p.tool || 'hand'} on ${block.name}`); } }
      return fail('No target for useOn');
    }

    // ═══ OBSERVATION (5) ═══
    case 'getInventory': return ok(JSON.stringify(bot.inventory.items().map(i => ({ name: i.name, count: i.count }))));
    case 'getNearbyBlocks': {
      const radius = p.radius || 16; const pos = bot.entity.position; const blocks = [];
      for (let dx = -radius; dx <= radius; dx += 2) for (let dy = -3; dy <= 3; dy += 2) for (let dz = -radius; dz <= radius; dz += 2) {
        const b = bot.blockAt(new Vec3(pos.x+dx, pos.y+dy, pos.z+dz));
        if (b && b.name !== 'air' && b.name !== 'cave_air') blocks.push({ name: b.name, x: pos.x+dx, y: pos.y+dy, z: pos.z+dz });
      }
      return ok(JSON.stringify(blocks.slice(0, 50)));
    }
    case 'getNearbyEntities': return ok(JSON.stringify(Object.values(bot.entities).filter(e => e !== bot.entity && e.position.distanceTo(bot.entity.position) < 32).map(e => ({ name: e.name || e.username, distance: e.position.distanceTo(bot.entity.position).toFixed(1) })).slice(0, 20)));
    case 'getHealth': return ok(JSON.stringify({ health: bot.health, food: bot.food, saturation: bot.foodSaturation, oxygen: bot.oxygenLevel }));
    case 'inspectWorld': return ok(JSON.stringify(getWorldState()));

    default: return fail(`Unknown action: ${type}`);
  }
}

function getWorldState() {
  const pos = bot.entity?.position; if (!pos) return { error: 'Not spawned' };
  const nearbyBlocks = [];
  for (let dx = -3; dx <= 3; dx++) for (let dy = -2; dy <= 2; dy++) for (let dz = -3; dz <= 3; dz++) {
    const b = bot.blockAt(new Vec3(pos.x+dx, pos.y+dy, pos.z+dz));
    if (b && b.name !== 'air' && b.name !== 'cave_air') nearbyBlocks.push({ name: b.name, x: pos.x+dx, y: pos.y+dy, z: pos.z+dz });
  }
  const nearbyEntities = Object.values(bot.entities).filter(e => e !== bot.entity && e.position.distanceTo(pos) < 32).map(e => ({ name: e.name || e.username, distance: e.position.distanceTo(pos).toFixed(1) }));
  return {
    position: { x: pos.x.toFixed(1), y: pos.y.toFixed(1), z: pos.z.toFixed(1) },
    health: bot.health, food: bot.food, saturation: bot.foodSaturation, oxygen: bot.oxygenLevel,
    gameMode: bot.gameMode, nearbyBlocks: nearbyBlocks.slice(0, 50), nearbyEntities: nearbyEntities.slice(0, 20),
    inventory: bot.inventory.items().map(i => ({ name: i.name, count: i.count })),
    timeOfDay: bot.time.timeOfDay, isRaining: bot.isRaining, biome: bot.blockAt(pos)?.biome || 'unknown',
  };
}

const http = require('http');
const server = http.createServer(async (req, res) => {
  res.setHeader('Content-Type', 'application/json');
  if (req.url === '/world' && req.method === 'GET') { res.end(JSON.stringify(getWorldState())); return; }
  if (req.url === '/status' && req.method === 'GET') {
    const pos = bot.entity?.position;
    res.end(JSON.stringify({ connected: !!bot.entity, username: bot.username, position: pos ? { x: pos.x.toFixed(1), y: pos.y.toFixed(1), z: pos.z.toFixed(1) } : null, health: bot.health, food: bot.food }));
    return;
  }
  if (req.url === '/action' && req.method === 'POST') {
    let body = ''; req.on('data', c => body += c); req.on('end', async () => {
      try { res.end(JSON.stringify(await executeAction(JSON.parse(body)))); } catch (e) { res.end(JSON.stringify({ success: false, message: e.message })); }
    }); return;
  }
  if (req.url === '/chat' && req.method === 'POST') {
    let body = ''; req.on('data', c => body += c); req.on('end', () => {
      bot.chat(JSON.parse(body).message); res.end(JSON.stringify({ success: true }));
    }); return;
  }
  res.statusCode = 404; res.end(JSON.stringify({ error: 'Not found' }));
});
const MF_PORT = parseInt(process.env.MF_PORT || '7001');
server.listen(MF_PORT, () => console.log(`Mineflayer API on port ${MF_PORT} — 41+ tools ready`));
'''
with open('/root/mineflayer_bot.js', 'w') as f:
    f.write(mineflayer_script)
print('✅ Mineflayer bot script v2 written — 41+ tools')

In [ ]:
# Cell 5: Start Mineflayer bot
import subprocess, os, time
os.environ['MC_SERVER'] = MC_SERVER.split(':')[0] if MC_SERVER else 'localhost'
os.environ['MC_PORT'] = MC_SERVER.split(':')[1] if ':' in MC_SERVER else '25565'
os.environ['MC_USERNAME'] = MC_USERNAME
os.environ['MC_VERSION'] = MC_VERSION
os.environ['MF_PORT'] = '7001'
os.environ['API_PORT'] = str(API_PORT)
print(f'⏳ Starting Mineflayer bot v2...')
mf_proc = subprocess.Popen(['node', '/root/mineflayer_bot.js'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=os.environ)
time.sleep(15)
output = mf_proc.stdout.read1(4096).decode() if mf_proc.stdout else ''
print(f'Output: {output[:500]}')
if 'SPAWNED' in output: print('✅ Bot spawned — 41+ tools available')
else: print('⏳ Waiting for spawn...')

In [ ]:
# Cell 6: FastAPI server — LLM agent loop v2 (41+ tools, fluid mode)
from fastapi import FastAPI
from pydantic import BaseModel
import requests, json, time, threading, re, subprocess, os, signal

app = FastAPI(title='Mineflayer LLM Agent v2', version='2.0.3')
MF_API = 'http://localhost:7001'
OLLAMA_API = f'{LLM_HOST}/v1/chat/completions'

class GoalRequest(BaseModel):
    goal: str
    max_actions: int = 80

class ChatRequest(BaseModel):
    message: str

class ConnectRequest(BaseModel):
    server: str  # host:port
    username: str = 'LLM_Bot'

SYSTEM_PROMPT = '''You are an AI agent controlling a Minecraft bot via Mineflayer with 41+ tools.
You receive the current world state and must decide the next actions.

## AVAILABLE ACTIONS (41+)
### MOVEMENT (8): moveTo{x,z}, followPlayer{name,distance?}, explore{radius?}, stop{}, lookAt{x,y,z}, jump{}, sprint{enable}, sneak{enable}
### MINING (4): collectBlocks{blockType,count}, digBlock{x,y,z}, mineResource{resource,count}, stripMine{direction,length}
### BUILDING (6): placeBlock{x,y,z,blockType}, buildHouse{size,material}, buildWall{length,height,material}, buildFloor{size,material}, towerUp{height,material}, bridge{length,material}
### CRAFTING (4): craft{item,count}, smelt{item,fuel,count}, enchant{item}, repair{item}
### COMBAT (5): attack{target}, hunt{}, defend{}, flee{}, jumpAttack{target}
### SURVIVAL (6): eat{}, sleep{}, fish{}, farm{}, tame{animal}, breed{animal}
### INVENTORY (6): equip{item,slot?}, unequip{slot?}, dropItem{item,count?}, depositItem{item,count?}, withdrawItem{item,count?}, sortInventory{}
### SOCIAL (2): chat{message}, trade{tradeIndex?}
### USE ON (1): useOn{tool,target?}
### OBSERVATION (5): getInventory{}, getNearbyBlocks{radius?}, getNearbyEntities{}, getHealth{}, inspectWorld{}

## RULES
1. Respond with a JSON array of actions: [{"type":"...","params":{...}}]
2. If health < 6, prioritize survival: eat, flee, or sleep.
3. For building: collect materials FIRST, then build.
4. For mining: moveTo near resource, then collectBlocks.
5. For combat: equip sword first, then attack.
6. Be efficient — batch related actions (max 5 per response).
7. If stuck, explore to find new resources.
8. Return EMPTY array [] when goal is achieved.
Respond with ONLY the JSON array.'''

# ─── World state cache (avoid spamming Mineflayer) ──────────────
_world_cache = None
_world_cache_time = 0

def get_world_state():
    global _world_cache, _world_cache_time
    now = time.time()
    if _world_cache and now - _world_cache_time < WORLD_CACHE_MS / 1000:
        return _world_cache
    try:
        _world_cache = requests.get(f'{MF_API}/world', timeout=8).json()
        _world_cache_time = now
        return _world_cache
    except Exception as e:
        return {'error': str(e)}

def execute_action(action):
    try:
        return requests.post(f'{MF_API}/action', json=action, timeout=60).json()
    except Exception as e:
        return {'success': False, 'message': str(e)}

def call_llm(world_state, goal, history=''):
    user_msg = f"""Goal: {goal}
World state: {json.dumps(world_state, indent=2)}
Previous actions: {history}
What actions next? JSON array only (empty [] if done)."""
    payload = {
        'model': LLM_MODEL,
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_msg},
        ],
        'temperature': 0.3,
        'max_tokens': 1000,
        'stream': False,
    }
    try:
        resp = requests.post(OLLAMA_API, json=payload, timeout=LLM_TIMEOUT)
        if resp.status_code == 200:
            content = resp.json()['choices'][0]['message']['content']
            json_match = re.search(r'\[.*\]', content, re.DOTALL)
            if json_match:
                return json.loads(json_match.group())
        return []
    except Exception as e:
        print(f'LLM error: {e}')
        return []

agent_running = False
agent_log = []
MAX_LOG_SIZE = 500  # Prevent unbounded memory growth

def add_log(entry):
    agent_log.append(entry)
    if len(agent_log) > MAX_LOG_SIZE:
        agent_log[:] = agent_log[-MAX_LOG_SIZE:]  # Trim in-place

# NOTE: mf_proc is set by Cell 5 — do NOT reassign here or we lose the handle
# mf_proc is imported from the global scope (Cell 5)

def run_agent_loop(goal, max_actions=80):
    global agent_running
    agent_running = True
    add_log(f'🎯 Goal: {goal}')
    history = ''
    step = 0  # Initialize before loop in case max_actions=0
    for step in range(max_actions):
        if not agent_running:
            add_log('⏹️ Stopped by user')
            break
        world = get_world_state()
        if 'error' in world:
            add_log(f'❌ World error: {world["error"]}')
            break
        # Auto-survival check
        hp = world.get('health', 20)
        if hp <= 4:
            add_log(f'⚠️ Low health ({hp}) — auto-survival mode')
            r1 = execute_action({'type': 'eat', 'params': {}})
            add_log(f'  eat: {r1.get("message","")}')
            r2 = execute_action({'type': 'flee', 'params': {}})
            add_log(f'  flee: {r2.get("message","")}')
            continue
        # Ask LLM for next actions
        actions = call_llm(world, goal, history)
        if not actions:
            add_log('✅ Goal complete or LLM returned no actions')
            break
        for action in actions:
            if not agent_running:
                break
            result = execute_action(action)
            desc = f'[{step}] {action["type"]}({json.dumps(action.get("params",{}))}): {result.get("message","")}'
            add_log(desc)
            history += f'\n{desc}'
            # Minimal delay for fluidity — only 100ms
            time.sleep(ACTION_DELAY_MS / 1000)
        # Trim history to avoid token overflow
        if len(history) > 4000:
            history = '(...earlier actions trimmed...)\n' + history[-2000:]
    agent_running = False
    add_log(f'🏁 Finished after {step + 1} steps')

def restart_mineflayer(server, username):
    """Kill and restart the Mineflayer bot process with new server config."""
    global mf_proc
    # Kill old process (mf_proc was set by Cell 5 or a previous /connect call)
    try:
        if mf_proc:
            mf_proc.terminate()
            mf_proc.wait(timeout=5)
    except:
        try: mf_proc.kill()
        except: pass
    # Also kill any leftover node process running mineflayer
    os.system('pkill -f mineflayer_bot 2>/dev/null')
    time.sleep(2)
    # Set env vars
    parts = server.split(':')
    os.environ['MC_SERVER'] = parts[0]
    os.environ['MC_PORT'] = parts[1] if len(parts) > 1 else '25565'
    os.environ['MC_USERNAME'] = username
    os.environ['MC_VERSION'] = MC_VERSION
    os.environ['MF_PORT'] = '7001'
    # Start new process
    mf_proc = subprocess.Popen(['node', '/root/mineflayer_bot.js'],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=os.environ)
    time.sleep(10)
    # Check if spawned
    output = mf_proc.stdout.read1(4096).decode() if mf_proc.stdout else ''
    if 'SPAWNED' in output:
        return {'success': True, 'message': f'Bot connected to {server} as {username}'}
    elif 'KICKED' in output:
        return {'success': False, 'message': f'Bot kicked from {server}: {output[:200]}'}
    elif 'ERROR' in output:
        return {'success': False, 'message': f'Connection error: {output[:200]}'}
    else:
        return {'success': True, 'message': f'Bot connecting to {server} as {username} (waiting for spawn)...'}

@app.get('/health')
async def health():
    return {'status': 'ok', 'llm': LLM_MODEL, 'mc': MC_SERVER, 'tools': '41+', 'version': '2.0.3'}

@app.get('/world')
async def world():
    return get_world_state()

@app.get('/status')
async def status():
    try:
        resp = requests.get(f'{MF_API}/status', timeout=8).json()
        resp['agent_running'] = agent_running
        resp['llm_model'] = LLM_MODEL
        resp['tools_count'] = '41+'
        resp['log_lines'] = len(agent_log)
        return resp
    except:
        return {'connected': False, 'agent_running': agent_running, 'log_lines': len(agent_log)}

@app.post('/connect')
async def connect_server(req: ConnectRequest):
    """Reconnect the bot to a different Minecraft server (hot-swap)."""
    global MC_SERVER, MC_USERNAME, _world_cache, _world_cache_time, agent_running
    # Stop agent if running
    if agent_running:
        agent_running = False
        time.sleep(1)
    # Update config
    MC_SERVER = req.server
    MC_USERNAME = req.username
    # Restart Mineflayer process
    result = restart_mineflayer(req.server, req.username)
    # Clear world cache
    _world_cache = None
    _world_cache_time = 0
    return result

@app.post('/goal')
async def set_goal(req: GoalRequest):
    global agent_running
    if agent_running:
        return {'error': 'Agent already running — send /stop first'}
    threading.Thread(target=run_agent_loop, args=(req.goal, req.max_actions), daemon=True).start()
    return {'success': True, 'goal': req.goal, 'max_actions': req.max_actions, 'tools': '41+'}

@app.post('/stop')
async def stop_agent():
    global agent_running
    agent_running = False
    try:
        requests.post(f'{MF_API}/action', json={'type': 'stop', 'params': {}}, timeout=5)
    except:
        pass
    return {'success': True, 'message': 'Agent stopped'}

@app.get('/log')
async def get_log(lines: int = 50):
    return {'log': '\n'.join(agent_log[-lines:]), 'total_lines': len(agent_log)}

@app.get('/log/since')
async def get_log_since(since: int = 0):
    """Get log lines since a given index — for incremental polling."""
    # Adjust since index if log was trimmed
    trim_offset = max(0, since - (len(agent_log) - MAX_LOG_SIZE)) if len(agent_log) >= MAX_LOG_SIZE else 0
    adjusted_since = max(0, since - trim_offset)
    new_lines = agent_log[adjusted_since:]
    return {'lines': new_lines, 'next_since': len(agent_log), 'total': len(agent_log)}

@app.post('/chat')
async def send_chat(req: ChatRequest):
    try:
        requests.post(f'{MF_API}/chat', json={'message': req.message}, timeout=10)
        return {'success': True}
    except Exception as e:
        return {'error': str(e)}

@app.post('/action')
async def send_action(action: dict):
    return execute_action(action)

print('✅ LLM Agent API v2.0.3 ready — 41+ tools, fluid mode, /connect hot-swap, log trimming')

In [ ]:
# Cell 7: Start ngrok + API server
from pyngrok import ngrok, conf
import nest_asyncio, threading, uvicorn
if NGROK_AUTHTOKEN: conf.get_default().auth_token = NGROK_AUTHTOKEN
ngrok.kill()
import time; time.sleep(2)
tunnel = ngrok.connect(API_PORT, 'http')
AGENT_URL = tunnel.public_url
print(f'🌐 LLM Agent v2 URL: {AGENT_URL}')
if BOT_WEBHOOK_URL:
    try: requests.post(BOT_WEBHOOK_URL, json={'url': AGENT_URL, 'type': 'mineflayer'}, timeout=10); print('📡 Bot notified')
    except Exception as e: print(f'⚠️ Webhook failed: {e}')
nest_asyncio.apply()
threading.Thread(target=lambda: uvicorn.run(app, host='0.0.0.0', port=API_PORT, log_level='info'), daemon=True).start()
time.sleep(3)
print(f'✅ API server v2 running on port {API_PORT}')

In [ ]:
# Cell 8: Keep-alive loop
import time, requests
from datetime import datetime, timedelta
start_time = datetime.now()
regenerate_at = start_time + timedelta(hours=24)
ok_count = 0; fail_count = 0
try:
    while True:
        now = datetime.now()
        try:
            resp = requests.get(f'{AGENT_URL}/health', timeout=10)
            if resp.status_code == 200: ok_count += 1
            else: fail_count += 1
        except: fail_count += 1
        elapsed = now - start_time
        if int(elapsed.total_seconds()) % 300 == 0 and int(elapsed.total_seconds()) > 0:
            try:
                st = requests.get(f'{AGENT_URL}/status', timeout=10).json()
                mc = f"MC={'online' if st.get('connected') else 'offline'} hp={st.get('health','?')} tools={st.get('tools_count','?')}"
            except: mc = 'MC=?'
            print(f'[{now.strftime("%H:%M:%S")}] up={int(elapsed.total_seconds()/60)}min ok={ok_count} fail={fail_count} {mc}')
        if now >= regenerate_at:
            print('🔄 Regenerating ngrok...')
            ngrok.kill(); time.sleep(3)
            AGENT_URL = ngrok.connect(API_PORT, 'http').public_url
            print(f'   New URL: {AGENT_URL}')
            if BOT_WEBHOOK_URL:
                try: requests.post(BOT_WEBHOOK_URL, json={'url': AGENT_URL, 'type': 'mineflayer'}); print('   📡 Bot notified')
                except: pass
            regenerate_at = now + timedelta(hours=24)
        time.sleep(60)
except KeyboardInterrupt: print('\n⏹️ Stopped')
except Exception as e: print(f'\n❌ {e}')
finally: print(f'Stats: ok={ok_count} fail={fail_count}')

In [ ]:
# Cell 9 (optional): Test
import requests
print('Health:', requests.get(f'{AGENT_URL}/health').json())
world = requests.get(f'{AGENT_URL}/world').json()
print(f'World: pos={world.get("position")} hp={world.get("health")} inv={len(world.get("inventory",[]))} entities={world.get("nearbyEntities",[])[:5]}')
# Test goals (uncomment):
# requests.post(f'{AGENT_URL}/goal', json={'goal': 'Collect 10 oak logs then craft planks', 'max_actions': 30})
# requests.post(f'{AGENT_URL}/goal', json={'goal': 'Build a 5x5 house', 'max_actions': 40})
# requests.post(f'{AGENT_URL}/goal', json={'goal': 'Hunt animals for food', 'max_actions': 15})
# requests.post(f'{AGENT_URL}/goal', json={'goal': 'Defend against zombies', 'max_actions': 20})
# requests.post(f'{AGENT_URL}/goal', json={'goal': 'Mine iron ore and smelt into ingots', 'max_actions': 50})